In [6]:
# ここで，.pyプログラムと configs/ が含まれていればよい．
%ls ./src

configs/              p1_0_sample_train.py         p4_0_output_train_result.py
p1_0_preprocess.py    p4_0_compute_cost.py
p1_0_sample_infer.py  p4_0_output_infer_result.py


In [7]:
# ここで，NN_config.json, preprocess_default_config.json, preprocess_sample_config.jsonなどのコンフィグが含まれていればよい．

%ls ./src/configs/1_0

NN_config.json               preprocess_default_config.json
no_cv_NN_config.json         preprocess_no_cv_sample_config.json
preprocess_config_manual.md  preprocess_sample_config.json


In [8]:
# datasetsは3.6.0推奨．4.0.0以降は動作しない．これらは，1_0_preprocess.pyを動かすために必要．
!pip install datasets==3.6.0 sudachipy pandas numpy pyarrow sudachidict_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 MB 12.3 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [13]:
# 左のバーのファイルから/content/src/1_0_preprocess.pyと/content/configs/<自分のディレクトリ>/<任意のconfigファイル>.jsonをアップロードして動作
# 今回はsrcをパッケージとして扱うので-mを使う
# 初回動作時は，Do you wish to run the custom code? [y/N]と聞かれるため，そのすぐ右をクリックしてyを入力
!python -m src.p1_0_preprocess --config ./src/configs/1_0/preprocess_sample_config.json

Generating train split: 100% 5894/5894 [00:00<00:00, 8188.96 examples/s]
Generating validation split: 100% 737/737 [00:00<00:00, 9880.15 examples/s]
Generating test split: 100% 736/736 [00:00<00:00, 9725.00 examples/s]
Start preprocessing...
config path: ./src/configs/1_0/preprocess_sample_config.json
Saved: {'folds': {'fold_1': {'train': 'src/data/1_0_preprocessed/cv_k5/fold_1/train.parquet', 'val': 'src/data/1_0_preprocessed/cv_k5/fold_1/val.parquet'}, 'fold_2': {'train': 'src/data/1_0_preprocessed/cv_k5/fold_2/train.parquet', 'val': 'src/data/1_0_preprocessed/cv_k5/fold_2/val.parquet'}, 'fold_3': {'train': 'src/data/1_0_preprocessed/cv_k5/fold_3/train.parquet', 'val': 'src/data/1_0_preprocessed/cv_k5/fold_3/val.parquet'}, 'fold_4': {'train': 'src/data/1_0_preprocessed/cv_k5/fold_4/train.parquet', 'val': 'src/data/1_0_preprocessed/cv_k5/fold_4/val.parquet'}, 'fold_5': {'train': 'src/data/1_0_preprocessed/cv_k5/fold_5/train.parquet', 'val': 'src/data/1_0_preprocessed/cv_k5/fold_5/val.

In [14]:
# ランタイムをGPUに変更すること．
# CPUでやると時間がかかる
# サンプルに時間をかけたくないのでEpoch数は少ない．本番は十分にEpoch数を確保すること
!python -m src.p1_0_sample_train --config ./src/configs/1_0/NN_config.json

[INFO] Random seed fixed to 42
[CV] root=src/data/1_0_preprocessed/cv_k5  folds=['fold_1', 'fold_2', 'fold_3', 'fold_4', 'fold_5']  has_test=True

===== FOLD 1/5 : fold_1 =====
Epoch 01 | tr_loss=3.5013 | tr_acc=0.5162 | va_loss=3.8812 | va_acc=0.4296
Epoch 02 | tr_loss=1.4553 | tr_acc=0.5573 | va_loss=2.1798 | va_acc=0.3612
Epoch 03 | tr_loss=1.0714 | tr_acc=0.6479 | va_loss=1.9835 | va_acc=0.3973
[FOLD 1] val_acc=0.4296 (best_tracker=0.4296)
[FOLD 1] test_acc=0.4130
[INFO] Config snapshot saved to src/result/1_0_preprocess_sample/fold_1/reports/2025-10-29_15-43-11/config/used_config.json
[val] accuracy=0.4296  macro_f1=0.4162  micro_f1=0.4296
[test] accuracy=0.4130  macro_f1=0.4019  micro_f1=0.4130
[TRAIN COST SUMMARY]
  - phase: train
  - started_at: 2025-10-29T15:42:49.749136+00:00
  - ended_at: 2025-10-29T15:43:10.578949+00:00
  - duration_sec: 20.8298
  - cuda_available: True
  - gpu_name: Tesla T4
  - gpu_mem_peak_mib: 408.6
  - pid: 2413
[INFO] Model saved to src/result/1_0_pre

Epoch数を十分に確保しても過学習する．  
より詳細な結果はresultディレクトリに出力している  
モデルとパラメータが適当なので結果は良くないが，少なくともコードの参考にはなるはず．  
過学習をしないように工夫してみてください．

In [15]:
# こっちは，クロスバリデーションをしないため，早い
# Epoch数が少ないので超早い
# configs/1_0にno_cvのconfigを入れてから実行
# 初回動作時は，Do you wish to run the custom code? [y/N]と聞かれるため，そのすぐ右をクリックしてyを入力
!python -m src.p1_0_preprocess --config ./src/configs/1_0/preprocess_no_cv_sample_config.json
!echo "start 1_0_sample.py"
!python -m src.p1_0_sample_train --config ./src/configs/1_0/no_cv_NN_config.json

Generating train split: 100% 5894/5894 [00:00<00:00, 8572.78 examples/s]
Generating validation split: 100% 737/737 [00:00<00:00, 10735.47 examples/s]
Generating test split: 100% 736/736 [00:00<00:00, 9457.74 examples/s]
Start preprocessing...
config path: ./src/configs/1_0/preprocess_no_cv_sample_config.json
Saved: {'train': 'src/data/1_0_no_cv_preprocessed/train.parquet', 'val': 'src/data/1_0_no_cv_preprocessed/val.parquet', 'test': 'src/data/1_0_no_cv_preprocessed/test.parquet'}
Generating train split: 5892 examples [00:00, 104952.45 examples/s]
Generating validation split: 736 examples [00:00, 98271.66 examples/s]
Generating test split: 739 examples [00:00, 106963.58 examples/s]
DatasetDict({
    train: Dataset({
        features: ['content', 'category'],
        num_rows: 5892
    })
    validation: Dataset({
        features: ['content', 'category'],
        num_rows: 736
    })
    test: Dataset({
        features: ['content', 'category'],
        num_rows: 739
    })
})

=== カテゴ

In [17]:
# 推論時の所要時間や使用VRAM量の計測
!python -m src.p1_0_sample_infer --ts-dir ./src/result/1_0_preprocess_sample/fold_1/reports/2025-10-29_15-43-11 --data ./src/data/1_0_no_cv_preprocessed/train.parquet --adapter bow --batch-size 256

[INFER COST SUMMARY]
  - phase: inference
  - started_at: 2025-10-29T15:50:36.541563+00:00
  - ended_at: 2025-10-29T15:50:37.113827+00:00
  - duration_sec: 0.5723
  - cuda_available: True
  - gpu_name: Tesla T4
  - gpu_mem_peak_mib: 79.9
  - pid: 4469
[INFER OUTPUT] saved under: /content/src/result/1_0_preprocess_sample/fold_1/reports/2025-10-29_15-43-11/inference_2025-10-29_15-50-37
[INFER] wall_time_sec: 0.5723
